# Exploratory data analysis

**Data**: Smart water meters — hourly register readings (Jan–Apr 2025)  
**Goal**: understand the raw data, identify quality issues, and select meters for the Box–Jenkins case studies.

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from utils import PROJECT_DIR, DATA_RAW, DATA_PROCESSED, OUTPUT_FIGURES, set_plot_style

set_plot_style()

# Load data
raw = pd.read_excel(os.path.join(DATA_RAW, "water_meters.xlsx"), engine="openpyxl")
raw.columns = raw.columns.str.strip()
hourly = pd.read_parquet(os.path.join(DATA_PROCESSED, "meters_hourly.parquet"))
daily = pd.read_parquet(os.path.join(DATA_PROCESSED, "meters_daily.parquet"))
metadata = pd.read_csv(os.path.join(DATA_PROCESSED, "meter_metadata.csv"))

print(f"Raw dataset: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"Date range: {raw['Sampling Date'].min()} to {raw['Sampling Date'].max()}")
print(f"Selected meters: {sorted(hourly['meter_id'].unique())}")
print(f"Hourly data: {hourly.shape[0]:,} rows | Daily data: {daily.shape[0]:,} rows")

## 1. Raw Dataset Overview

"The dataset contains hourly smart meter readings from 715 domestic meters covering approximately January–April 2025. Each record provides the instantaneous water consumption in m³/h along with household metadata."

In [ ]:
# Identify meters by timestamp resets
dates = raw["Sampling Date"].values
breaks = [0]
for i in range(1, len(dates)):
    if dates[i] <= dates[i - 1]:
        breaks.append(i)

meter_ids_raw = np.zeros(len(raw), dtype=int)
for idx in range(len(breaks)):
    start = breaks[idx]
    end = breaks[idx + 1] if idx + 1 < len(breaks) else len(raw)
    meter_ids_raw[start:end] = idx
raw["meter_id"] = meter_ids_raw

# Meter-level statistics
meter_stats = raw.groupby("meter_id").agg(
    n_records=("Value", "count"),
    mean_val=("Value", "mean"),
    zero_frac=("Value", lambda x: (x == 0).mean()),
    date_min=("Sampling Date", "min"),
    date_max=("Sampling Date", "max"),
).reset_index()
meter_stats["span_days"] = (meter_stats["date_max"] - meter_stats["date_min"]).dt.total_seconds() / 86400

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(meter_stats["n_records"], bins=50, edgecolor="k", alpha=0.7)
axes[0].set_xlabel("Number of records"); axes[0].set_ylabel("Meters"); axes[0].set_title("Records per meter")
axes[0].axvline(1000, color="red", linestyle="--", label="Selection threshold (1000)")
axes[0].legend(fontsize=8)

axes[1].hist(meter_stats["zero_frac"] * 100, bins=40, edgecolor="k", alpha=0.7, color="orange")
axes[1].set_xlabel("Zero fraction (%)"); axes[1].set_ylabel("Meters"); axes[1].set_title("Zero-consumption fraction")

axes[2].hist(meter_stats["span_days"], bins=40, edgecolor="k", alpha=0.7, color="green")
axes[2].set_xlabel("Coverage span (days)"); axes[2].set_ylabel("Meters"); axes[2].set_title("Temporal coverage")

fig.suptitle(f"Raw Dataset: {len(breaks)} meters, {raw.shape[0]:,} readings", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_raw_overview.png"), dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_raw_overview.pdf"), dpi=300, bbox_inches="tight")
plt.show()

## 2. Selected Meters

We select 10 meters based on: (i) >1000 records, (ii) >80% temporal coverage, (iii) <80% zero-consumption fraction. Priority is given to meters with full January–April coverage (~2400 hours). The selected meters span a range of annual consumption levels to ensure diversity.

In [ ]:
# Selected meters metadata
display(metadata[["meter_id", "n_records", "coverage", "mean_consumption", "std_consumption",
                  "zero_fraction", "annual_consumption"]].round(4))

# Time series for all selected meters
meter_list = sorted(hourly["meter_id"].unique())
n = len(meter_list)
fig, axes = plt.subplots(n, 1, figsize=(14, 2.2 * n), sharex=False)
for i, mid in enumerate(meter_list):
    m = hourly[hourly["meter_id"] == mid].set_index("timestamp").sort_index()
    axes[i].plot(m.index, m["consumption"], linewidth=0.4)
    axes[i].set_ylabel(f"Meter {mid}", fontsize=9)
    axes[i].tick_params(labelsize=8)
axes[-1].set_xlabel("Date")
fig.suptitle("Hourly Water Consumption — Selected Meters (m³/h)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_all_meters_hourly.png"), dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_all_meters_hourly.pdf"), dpi=300, bbox_inches="tight")
plt.show()

## 3. Distribution of Consumption Values

Water consumption data is heavily **zero-inflated**: many hours have zero flow, especially during nighttime. The distribution is right-skewed with occasional high-consumption spikes. This motivates the use of Box-Cox transformations for heteroscedastic meters.

In [ ]:
# Distribution of hourly consumption (selected meters)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Overall distribution
vals = hourly["consumption"]
axes[0, 0].hist(vals[vals > 0], bins=80, edgecolor="k", alpha=0.7, log=True)
axes[0, 0].set_xlabel("Hourly consumption (m³/h)"); axes[0, 0].set_ylabel("Count (log)")
axes[0, 0].set_title("Distribution (excluding zeros)")

# Zero fraction per meter
mids = sorted(hourly["meter_id"].unique())
zero_fracs = [((hourly[hourly["meter_id"] == m]["consumption"] == 0).mean() * 100) for m in mids]
axes[0, 1].barh([str(m) for m in mids], zero_fracs, color="coral", edgecolor="k")
axes[0, 1].set_xlabel("Zero fraction (%)"); axes[0, 1].set_title("Zero-consumption fraction per meter")

# Boxplot per meter (daily totals)
daily_pivot = daily.pivot(index="date", columns="meter_id", values="daily_consumption")
daily_pivot.boxplot(ax=axes[1, 0], rot=45)
axes[1, 0].set_ylabel("Daily consumption (m³/day)"); axes[1, 0].set_title("Daily consumption boxplot")

# Summary statistics table
summary = hourly.groupby("meter_id")["consumption"].describe().round(4)
axes[1, 1].axis("off")
table = axes[1, 1].table(cellText=summary[["mean", "std", "50%", "max"]].values.round(4),
                          rowLabels=[str(m) for m in summary.index],
                          colLabels=["Mean", "Std", "Median", "Max"],
                          loc="center", cellLoc="center")
table.auto_set_font_size(False); table.set_fontsize(9)
axes[1, 1].set_title("Hourly consumption statistics (m³/h)")

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_distributions.png"), dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_distributions.pdf"), dpi=300, bbox_inches="tight")
plt.show()

## 4. Temporal Patterns

Domestic water consumption exhibits clear **intra-day** (morning and evening peaks) and **intra-week** (lower weekends) patterns. These cyclical patterns motivate the seasonal component in our SARIMA models.

In [ ]:
# Hourly profile (average consumption by hour of day)
hourly_ts = hourly.copy()
hourly_ts["hour"] = pd.to_datetime(hourly_ts["timestamp"]).dt.hour
hourly_ts["dow"] = pd.to_datetime(hourly_ts["timestamp"]).dt.dayofweek  # 0=Mon, 6=Sun

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Hourly profile (all meters averaged)
hourly_profile = hourly_ts.groupby("hour")["consumption"].mean()
axes[0].bar(hourly_profile.index, hourly_profile.values, color="steelblue", edgecolor="k", alpha=0.8)
axes[0].set_xlabel("Hour of day"); axes[0].set_ylabel("Mean consumption (m³/h)")
axes[0].set_title("Average hourly profile (all meters)")
axes[0].set_xticks(range(0, 24, 3))

# Weekly profile (daily totals by day of week)
daily_ts = daily.copy()
daily_ts["dow"] = pd.to_datetime(daily_ts["date"]).dt.dayofweek
dow_profile = daily_ts.groupby("dow")["daily_consumption"].mean()
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
axes[1].bar(range(7), dow_profile.values, color="coral", edgecolor="k", alpha=0.8)
axes[1].set_xticks(range(7)); axes[1].set_xticklabels(dow_labels)
axes[1].set_ylabel("Mean daily consumption (m³/day)")
axes[1].set_title("Average weekly profile (all meters)")

# Heatmap: consumption by hour and day of week
hourly_heatmap = hourly_ts.groupby(["dow", "hour"])["consumption"].mean().unstack()
im = axes[2].imshow(hourly_heatmap.values, aspect="auto", cmap="YlOrRd", interpolation="nearest")
axes[2].set_xlabel("Hour of day"); axes[2].set_ylabel("")
axes[2].set_yticks(range(7)); axes[2].set_yticklabels(dow_labels)
axes[2].set_xticks(range(0, 24, 3))
axes[2].set_title("Consumption heatmap (hour x weekday)")
plt.colorbar(im, ax=axes[2], label="m³/h", shrink=0.8)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_temporal_patterns.png"), dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_temporal_patterns.pdf"), dpi=300, bbox_inches="tight")
plt.show()

## 5. Daily Consumption Time Series

The daily aggregated series is the primary input for our SARIMA analysis, with weekly seasonality ($m = 7$). We model daily totals rather than hourly data because: (i) the hourly series is heavily zero-inflated, (ii) daily aggregation provides smoother series amenable to Box-Jenkins methodology, and (iii) weekly patterns are the dominant seasonal component at this resolution.

In [ ]:
# Daily consumption for a subset of representative meters
showcase = [183, 345, 503, 575]  # diverse consumption levels
fig, axes = plt.subplots(len(showcase), 1, figsize=(14, 3 * len(showcase)), sharex=False)
for i, mid in enumerate(showcase):
    m = daily[daily["meter_id"] == mid].set_index("date").sort_index()
    axes[i].plot(m.index, m["daily_consumption"], "o-", markersize=2, linewidth=0.8)
    axes[i].axhline(m["daily_consumption"].mean(), color="red", linestyle="--", alpha=0.5,
                     label=f"Mean = {m['daily_consumption'].mean():.3f} m³/day")
    axes[i].set_ylabel(f"Meter {mid}\n(m³/day)", fontsize=10)
    axes[i].legend(fontsize=9, loc="upper right")
axes[-1].set_xlabel("Date")
fig.suptitle("Daily Water Consumption — Representative Meters", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_daily_series.png"), dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_daily_series.pdf"), dpi=300, bbox_inches="tight")
plt.show()

## 6. Correlation Between Meters

We examine cross-correlation between meter consumption patterns to understand whether meters in the same network exhibit similar behavior.

In [ ]:
# Correlation matrix of daily consumption
corr = daily_pivot.corr()
fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Cross-correlation of daily consumption between meters")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_correlation.png"), dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_FIGURES, "eda_correlation.pdf"), dpi=300, bbox_inches="tight")
plt.show()
print(f"Mean pairwise correlation: {corr.where(~np.eye(len(corr), dtype=bool)).mean().mean():.3f}")

## 7. Key EDA Findings

1. **Dataset size**: 715 meters, 409K hourly readings covering ~101 days (Jan–Apr 2025)
2. **Zero-inflation**: ~40% of hourly readings are zero — nighttime hours typically show no consumption
3. **Temporal patterns**: Clear morning (7–9 AM) and evening (7–9 PM) peaks; lower consumption on weekends
4. **Heteroscedasticity**: Some meters (141, 202, 303, 575) show variance proportional to level → Box-Cox transformation needed
5. **Selected meters**: 10 meters with diverse consumption levels (42–989 m³/year), 6 with full 101-day coverage, 4 with ~54-day coverage
6. **Low inter-meter correlation**: Meters behave independently, justifying per-meter SARIMA modeling

These findings inform our modeling strategy: daily-aggregated SARIMA($p,d,q$)($P,D,Q$)$_7$ with Box-Cox transformation where needed.